# Frontier estimation on Colab

Serves a frozen open-weight model with vLLM and runs the frontier
estimation pipeline against it.

**Before running anything expensive, read this:**

- Colab preempts sessions. The sampler appends to `fsync`-ed JSONL shards and
  resumes where it stopped, so an interrupted run is recoverable — *provided*
  the output directory is on Drive (cell 2). Output left on local disk is lost
  when the runtime recycles.
- Colab runtimes are containers and cannot nest Docker, so verification runs on
  the subprocess backend, which is **not a security boundary**. That is
  acceptable for frontier sampling of a small frozen model on curated coding
  tasks. It is **not** acceptable for running `S_evo` self-modifying scaffolds
  (see `docs/DECISIONS.md` D-02).
- Always `--dry-run` first. It reports the sample count without spending GPU time.

**Sizing.** `N_max=1000` over ~100 tasks is ~200k generations, roughly 5–6
T4-hours for a 1.5B model, and bounds a zero-success task at `p < 0.003`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

# T4 (16GB) comfortably serves a 1.5B model in fp16 and a 7B model quantised.
# T4 is Turing and has no bf16, so dtype must be float16 below.
# If this cell shows no GPU: Runtime > Change runtime type > T4 GPU.

## 2. Mount Drive for checkpoints

Skip this and a preempted run loses its shards.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUN_DIR = '/content/drive/MyDrive/cbs/runs/frontier_qwen15b'
os.makedirs(RUN_DIR, exist_ok=True)
print('checkpoints ->', RUN_DIR)

## 3. Install

In [ ]:
# Point at your fork/clone.
REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO.git'

import os
if not os.path.exists('/content/benchmark'):
    !git clone $REPO_URL /content/benchmark
%cd /content/benchmark
!pip install -q -e ".[serving,dev]"
!pip install -q vllm

!cbs env

## 4. Validate the instrument before spending GPU time

Runs entirely on the mock backend against known ground truth. If this fails,
the estimates from a real model would be untrustworthy too — fix it first.

In [ ]:
!cbs tasks verify
!cbs frontier validate --n-samples 200

## 5. Serve the frozen model

In [ ]:
import subprocess, time, requests

MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--dtype', 'float16',            # T4 has no bf16
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.85',
    '--port', '8000',
    '--disable-log-requests',
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for attempt in range(120):
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).status_code == 200:
            print('vLLM ready'); break
    except Exception:
        pass
    if server.poll() is not None:
        raise RuntimeError('vLLM exited early; check the logs cell below')
    time.sleep(5)
else:
    raise TimeoutError('vLLM did not come up within 10 minutes')

## 6. Freeze the splits

Splits must be frozen and hashed **before** any results exist against them
(brief §7, Phase 1 DoD). The suite hash is quoted in the results.

In [ ]:
!cbs splits freeze --family toy --train 0.4 --held-out 0.4 --transfer 0.2 \
    --out /content/drive/MyDrive/cbs/splits/toy.json
!cbs splits verify --family toy --manifest /content/drive/MyDrive/cbs/splits/toy.json

## 7. Dry run — confirm the budget before spending it

In [ ]:
!cbs frontier estimate --config configs/frontier_vllm_colab.yaml \
    --output-dir $RUN_DIR --dry-run

## 8. Run

Safe to re-run after a preemption: it resumes from the shards on Drive.
Start small (`--n-max 50`) to confirm the whole path works against the real
model, then raise it.

In [ ]:
!cbs frontier estimate --config configs/frontier_vllm_colab.yaml \
    --output-dir $RUN_DIR --n-max 50

## 9. Inspect the records

In [ ]:
import json, pathlib

records = [json.loads(l) for l in
           (pathlib.Path(RUN_DIR) / 'records.jsonl').read_text().splitlines() if l.strip()]

for r in records:
    if r['beyond_frontier']:
        print(f"{r['task_id']:28s} BEYOND-FRONTIER  {r['beyond_frontier_qualifier']}")
    else:
        print(f"{r['task_id']:28s} p_hat={r['p_hat']:.4f} "
              f"[{r['p_ci_low']:.4f},{r['p_ci_high']:.4f}]  "
              f"distinct={r['n_distinct_solutions']}  "
              f"coverage={r['sample_coverage']}  "
              f"saturated={r['solution_set_saturated']}")

# Unsaturated tasks are the ones whose frontier estimate is visibly
# budget-limited. Any crossing claim about them must carry that caveat.
unsat = [r['task_id'] for r in records
         if not r['beyond_frontier'] and not r['solution_set_saturated']]
print('\nsolution set still growing at N_max:', unsat)

## Debug: vLLM logs

In [ ]:
# server.terminate()  # free the GPU when done
import itertools
print(''.join(itertools.islice(server.stdout, 60)))